# 06 — Failure Analysis & Consolidated Results (Deliverable 2)

Deep-dive into four failure modes identified from the baseline run, followed by a master cross-experiment results table and cross-experiment insights.

**Dependencies:** run `04_baseline_evaluation.ipynb` and `05_experiments.ipynb` first — this notebook loads their cached result files from `data/eval/results/`.

In [ ]:
import sys, json, textwrap
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from . import eval as fe

baseline = json.load(open(fe.RESULTS_DIR / 'baseline.json', encoding='utf-8'))
print('✅ Harness imported')
print(f'   Baseline loaded: {baseline["aggregate"]["n_questions"]} questions')

✅ Harness imported
   Baseline loaded: 30 questions


---
## Failure Analysis

The brief requires four failure types, each with an **example**, **explanation**, and **attempted fix**. All examples are drawn from the 30-question gold set run against the baseline configuration (Groq `llama-3.3-70b` generator, `gpt-oss-120b` judge) unless noted.

| # | Failure type | Gold-set question | Root cause |
|---|---|---|---|
| 1 | Retrieval failure | Q06 — NSCA youth safety | Wrong NSCA doc retrieved |
| 2 | Hallucination | Q11 — WHO partial-guideline advice | Model supplements context from training data |
| 3 | Ambiguous query | Q22 — "Is more training always better?" | Short query matches too broadly |
| 4 | Irrelevant context | Q20 — NSCA LTAD recovery | Correct source drowned by 4 irrelevant chunks → wrong refusal |

In [2]:
pq = {r['id']: r for r in baseline['per_question']}

CASES = [
    ('Q06', 'Failure type 1: Retrieval failure'),
    ('Q11', 'Failure type 2: Hallucination'),
    ('Q22', 'Failure type 3: Ambiguous query'),
    ('Q20', 'Failure type 4: Irrelevant context -> wrong refusal'),
]

for qid, label in CASES:
    r = pq[qid]
    print(f'{"=" * 68}')
    print(f'  {label}  [{qid}]')
    print(f'{"=" * 68}')
    print(f'  Question  : {r["question"]}')
    print(f'  Category  : {r["category"]}')
    print(f'  Expected  : {r["expected_sources"]}')
    print(f'  Retrieved : {r["retrieved_sources"]}')
    print(f'  Recall@k  : {r.get("recall_at_k", "N/A")}   Hit: {r.get("hit", "N/A")}')
    print(f'  Refused   : {r["refused"]}')
    if r.get('faithfulness') is not None:
        print(f'  Faithfulness: {r["faithfulness"]}  - {r.get("faithfulness_reason","")}')
    if r.get('correctness') is not None:
        print(f'  Correctness : {r["correctness"]}  - {r.get("correctness_reason","")}')
    ans = (r.get('answer') or '').strip()
    print(f'  Answer    : {textwrap.shorten(ans, 220)}')
    print()

  Failure type 1: Retrieval failure  [Q06]
  Question  : Is resistance training safe for children and adolescents according to the NSCA youth position statement?
  Category  : in_scope
  Expected  : ['NSCA_2.pdf']
  Retrieved : ['NSCA_5.pdf', 'NSCA_4.pdf', 'NSCA_5.pdf', 'NSCA_5.pdf', 'NSCA_5.pdf']
  Recall@k  : 0.0   Hit: 0.0
  Refused   : False
  Faithfulness: 1.0  - All claims are directly supported by the cited sources.
  Correctness : 1.0  - Accurately states that properly designed, supervised resistance training is safe for youth, matching the reference.
  Answer    : According to the NSCA youth position statement, a properly designed and supervised resistance training program is relatively safe for youth. The risk of injury can be minimized by limiting the number of heavy lifts [...]

  Failure type 2: Hallucination  [Q11]
  Question  : Does the WHO recommend any physical activity even for people who cannot meet the full guidelines?
  Category  : in_scope
  Expected  : ['WHO.pdf'

### Failure type 1 — Retrieval failure (Q06)

**Example.**
> *"Is resistance training safe for children and adolescents according to the NSCA youth position statement?"*
> Expected source: `NSCA_2.pdf` (the NSCA youth resistance training position statement).
> Retrieved: `NSCA_5.pdf` (×4), `NSCA_4.pdf` (×1) — `NSCA_2.pdf` was never retrieved. Recall@5 = 0.000.

**Explanation.** The query embeds semantically close to all NSCA documents because they share vocabulary (resistance training, safety, youth, NSCA). The embedding model cannot distinguish between the five NSCA PDFs on the basis of document identity — it retrieves whichever chunks score highest on cosine similarity. This is a *vocabulary collision* failure: the query's intent (a specific document) cannot be resolved by dense semantic search alone.

**Outcome.** Despite the retrieval miss, the 70B generator answered correctly (correctness = 1.0, faithfulness = 1.0) because NSCA_5 also contains youth-relevant safety content. This is a *silent retrieval failure* — the system appears correct but cites the wrong source.

**Attempted fix.** Two approaches tested via H1:
1. **Larger k (k=10):** Hit-rate rises from 0.917 to 0.958. Q06 specifically is not resolved (the cosine gap is large), but this fixes borderline misses.
2. **Similarity search:** At k=5, overall recall is marginally higher (0.882 vs 0.868). Neither closes the vocabulary-collision gap for Q06. A hybrid keyword+semantic search (BM25 + dense) would be the principled fix — the document title "youth position statement" contains discriminating keywords that BM25 would weight heavily.

---

### Failure type 2 — Hallucination (Q11)

**Example.**
> *"Does the WHO recommend any physical activity even for people who cannot meet the full guidelines?"*
> Retrieved: all 5 chunks from `WHO.pdf` — perfect retrieval (Recall@5 = 1.0, Hit = 1.0).
> Faithfulness = **0.5** (70B generator). Judge reason: *"One claim about recommending activity even when full guidelines can't be met is not supported by the provided context."*

**Explanation.** Retrieval was flawless — every chunk came from the right document. The failure occurred at generation: the 70B model *knows* WHO guidelines from its pretraining data and adds a claim that is factually correct but not present in the specific retrieved chunks. This is the textbook hallucination pattern in RAG: the model fills in gaps from training knowledge, producing an answer that *reads* grounded but contains at least one unverifiable claim.

**Outcome.** The answer scored correctness = 1.0 (factually accurate) but faithfulness = 0.5 (partially ungrounded). For a trusted health/fitness assistant, an unverifiable claim is a liability even if it happens to be correct.

**Attempted fix.** H4 directly tested this: the strict prompt reduced hallucination measurably — **faithfulness 0.833 (strict) vs 0.717 (lenient)**. The strict prompt is already the baseline. The residual hallucination at faithfulness = 0.5 for Q11 suggests the 70B model occasionally overrides the instruction when highly confident in its training knowledge. A stronger fix would be a post-generation faithfulness check (claim-by-claim verification against the context).

---

### Failure type 3 — Ambiguous query handling (Q22)

**Example.**
> *"Is more training always better?"*
> Expected sources: `NSCA_4.pdf` (LTAD, overtraining section) + `SSW.pdf` (Starting Strength, periodisation).
> Retrieved: `NSCA_2.pdf`, `ProgressiveOverload...pdf`, `WHO.pdf`, `NSCA_5.pdf` (×2). Recall@5 = 0.000.

**Explanation.** The query is six words long and open-ended. Its embedding vector lands in a region of the semantic space shared by many fitness documents. The intended sources are not the closest cosine neighbours to such a short, general query. This is a *semantic underspecification* failure: the query does not carry enough signal to discriminate between documents.

**Outcome.** Despite Recall@5 = 0.000, the 70B model produced a correct answer (correctness = 1.0, faithfulness = 1.0) by reasoning from tangentially related chunks. This is a silent failure — right answer for the wrong reasons.

**Attempted fix.** No single-axis experiment fully addresses this. The most direct fix is **query expansion** before embedding: ask the LLM to rewrite the short query into a longer, more specific form before embedding (HyDE or step-back prompting). As a simpler workaround, an **ambiguous query classifier** at the front of the pipeline could prompt the user for clarification before retrieving.

---

### Failure type 4 — Irrelevant context (Q20)

**Example.**
> *"What recovery considerations does the NSCA LTAD position statement emphasize?"*
> Expected source: `NSCA_4.pdf` (LTAD = Long-Term Athlete Development).
> Retrieved: `NSCA_2.pdf`, `NSCA_1.pdf`, `NSCA_3.pdf`, **`NSCA_4.pdf`**, `NSCA_1.pdf` — the correct source appears at rank 4 but is surrounded by 4 chunks from unrelated NSCA documents.
> Result: **wrong refusal** (refused = True, correctness = 0.0).

**Explanation.** This is the inverse of a pure retrieval miss: the correct source *was* retrieved (Recall@5 = 1.0, Hit = 1.0) but appears at rank 4 among mostly irrelevant NSCA chunks. The generator, faced with a context dominated by off-topic NSCA content and only one weak signal about LTAD recovery, concluded that the context was insufficient and triggered the refusal sentence. Correct retrieval at the document level masked a chunk-level mismatch.

**Outcome.** The model refused a question for which it had the right source document — a false negative. The user gets no answer despite the knowledge existing in the knowledge base.

**Attempted fix.** Two experiment findings bear on this:
1. **Larger chunk size (H2):** Chunk 1024 raised MRR from 0.771 to 0.792 — a larger NSCA_4 chunk is more likely to contain both the document-level context and the specific recovery section.
2. **Higher λ in MMR (H1):** λ=0.8 raised Precision@k from 0.575 to 0.650 — a more relevance-weighted MMR would be less likely to diversity-select four chunks from different NSCA docs.
Combining chunk_size=1024 with MMR λ=0.8 would reduce the noise-to-signal ratio in the context.

---
## Consolidated Results Table

All experiments in one place. Retrieval metrics are objective (no judge). Answer-quality metrics use the **local `qwen2.5:7b` judge throughout** for cross-experiment comparability — the same judge, same generator (`llama3.1:8b`), only one axis changed per row. The baseline row is `h2_cs512` (MMR k=5, chunk 512, multi-qa-MiniLM, strict prompt, local pipeline).

In [3]:
def load_agg(fname):
    p = fe.RESULTS_DIR / fname
    return json.load(open(p, encoding='utf-8'))['aggregate']

# Load sweep data
h1 = json.load(open(fe.RESULTS_DIR / 'h1_retrieval_sweep.json', encoding='utf-8'))['results']
h1_by_name = {r['name']: r for r in h1}

h2_retr = json.load(open(fe.RESULTS_DIR / 'h2_chunksize_sweep.json', encoding='utf-8'))['results']
h2_by_cs = {r['chunk_size']: r for r in h2_retr}

h3_retr = json.load(open(fe.RESULTS_DIR / 'h3_embedding_sweep.json', encoding='utf-8'))['results']
h3_by_model = {r['model']: r for r in h3_retr}

def row(experiment, config, varied, retr_src, aq_fname=None):
    r = {
        'Experiment': experiment,
        'Configuration': config,
        'Varied axis': varied,
        'Recall@5 (in_scope)': round(retr_src.get('recall_at_k_in_scope', 0), 3),
        'Precision@5':         round(retr_src.get('precision_at_k', 0), 3),
        'MRR':                 round(retr_src.get('mrr', 0), 3),
        'Hit-rate':            round(retr_src.get('hit_rate', 0), 3),
        'Correctness':         '-',
        'Faithfulness':        '-',
        'Refusal acc.':        '-',
    }
    if aq_fname:
        a = load_agg(aq_fname)
        r['Correctness']  = round(a['correctness_in_scope'], 3) if a['correctness_in_scope'] is not None else '-'
        r['Faithfulness'] = round(a['faithfulness'], 3)         if a['faithfulness']         is not None else 'N/A'
        r['Refusal acc.'] = round(a['refusal_accuracy'], 3)     if a['refusal_accuracy']     is not None else '-'
    return r

rows = [
    row('Baseline',       'MMR k=5, cs512, multi-qa-MiniLM, strict', '-',
        h1_by_name['mmr_k5'], 'h2_cs512.json'),
    row('H1 — Retrieval', 'Similarity k=5',    'retrieval strategy', h1_by_name['sim_k5']),
    row('H1 — Retrieval', 'MMR k=3',           'retrieval strategy', h1_by_name['mmr_k3']),
    row('H1 — Retrieval', 'MMR k=10',          'retrieval strategy', h1_by_name['mmr_k10']),
    row('H1 — Retrieval', 'MMR lambda=0.8 k=5','retrieval strategy', h1_by_name['mmr_k5_lam0.8']),
    row('H2 — Chunk size','chunk=256, overlap=50',  'chunk size', h2_by_cs[256],  'h2_cs256.json'),
    row('H2 — Chunk size','chunk=512, overlap=50',  'chunk size (baseline)', h2_by_cs[512],  'h2_cs512.json'),
    row('H2 — Chunk size','chunk=1024, overlap=50', 'chunk size', h2_by_cs[1024], 'h2_cs1024.json'),
    row('H3 — Embedding', 'multi-qa-MiniLM (baseline)', 'embedding model',
        h3_by_model['multi-qa-MiniLM (baseline)'], 'h2_cs512.json'),
    row('H3 — Embedding', 'all-MiniLM-L6-v2',  'embedding model',
        h3_by_model['all-MiniLM-L6-v2'],  'h3_allMiniLM.json'),
    row('H3 — Embedding', 'bge-small-en-v1.5', 'embedding model',
        h3_by_model['bge-small-en-v1.5'], 'h3_bge.json'),
    row('H4 — Prompt',    'strict (baseline)', 'prompt variant',
        h1_by_name['mmr_k5'], 'h2_cs512.json'),
    row('H4 — Prompt',    'lenient',           'prompt variant',
        h1_by_name['mmr_k5'], 'h4_lenient.json'),
    row('RAG vs LLM',     'RAG — baseline',    'architecture',
        h1_by_name['mmr_k5'], 'h2_cs512.json'),
    row('RAG vs LLM',     'Direct LLM (no retrieval)', 'architecture',
        {'recall_at_k_in_scope': 0, 'precision_at_k': 0, 'mrr': 0, 'hit_rate': 0},
        'direct_llm.json'),
]

master = pd.DataFrame(rows).set_index(['Experiment', 'Configuration'])
pd.set_option('display.max_colwidth', 32)
pd.set_option('display.width', 120)
print(master.to_string())

                                                                   Varied axis  Recall@5 (in_scope)  Precision@5    MRR  Hit-rate Correctness Faithfulness Refusal acc.
Experiment      Configuration                                                                                                                                          
Baseline        MMR k=5, cs512, multi-qa-MiniLM, strict                      -                0.925        0.575  0.771     0.917       0.637        0.833         0.92
H1 — Retrieval  Similarity k=5                              retrieval strategy                0.925        0.642  0.778     0.917           -            -            -
                MMR k=3                                     retrieval strategy                0.825        0.583  0.750     0.833           -            -            -
                MMR k=10                                    retrieval strategy                0.925        0.562  0.776     0.958           -            -      

### Cross-experiment insights

**What each experiment axis contributed:**

| Axis | Winner | Key number | Conclusion |
|---|---|---|---|
| H1 Retrieval strategy | Similarity (retrieval) / tied (AQ) | Sim Precision@5 0.642 vs MMR 0.575 | MMR offers no advantage; similarity is the simpler, stronger default |
| H2 Chunk size | 512 (recall), 1024 (precision + faithfulness) | Faithfulness 0.716→0.833→0.838 | 256 is actively harmful; 512 maximises recall; 1024 maximises faithfulness |
| H3 Embedding | multi-qa-MiniLM (retrieval) | Recall@5 0.925 vs 0.875 for others | QA-trained embedding beats general-purpose; baseline justified |
| H4 Prompt | Strict | Faithfulness 0.833 vs 0.717; OOS refused 1.000 vs 0.000 | Strict is non-negotiable for safety and grounding |
| RAG vs LLM | RAG (safety + grounding) | OOS refused 1.000 (RAG) vs 0.000 (direct) | Direct LLM has higher correctness but answers everything — unsafe for a scoped assistant |

**Best single-axis improvement identified:** chunk_size=1024 raises faithfulness from 0.833 → 0.838 at equal hit-rate (0.917), with better MRR (0.792 vs 0.771). Combined with similarity retrieval (higher precision), this is the strongest evidence-backed alternative to the baseline.

**What no single axis fixed:** answer correctness remains in the 0.6–0.7 range across all configurations with the 8B local generator. The baseline 70B generator (Groq, `faithfulness=0.978`) shows that **generator size is the largest lever on both faithfulness and correctness** — more than any retrieval or chunking choice tested here.

**The baseline is well-chosen:** three of four hypotheses (H1, H2, H3) were rejected — the baseline configuration holds up under systematic testing. H4 is the only confirmed improvement, and it validates an existing design choice (strict prompt was already the baseline). The experiments provide quantitative justification for every component of the baseline rather than identifying a superior alternative.